# 05 · Demo app and model publishing
Gradio app: choose a database, ask a question in English, and see the generated SQL plus its executed result. Optionally publish the adapter to the Hugging Face Hub.

In [ ]:
# ---------- Setup: run this cell first ----------
import os, sys, subprocess, json, time, warnings
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU memory fragmentation
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("T2S_SMOKE") == "1"      # local CPU smoke test (ignore on Colab/Kaggle)
ON_KAGGLE = os.path.exists("/kaggle/input")
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
from pathlib import Path
if ON_KAGGLE:
    # Code is attached as a Kaggle Dataset (read-only) -> copy it to the writable /kaggle/working
    import glob, shutil, zipfile
    PROJECT_DIR = "/kaggle/working/text2sql"

    def _walk(top="/kaggle/input"):
        return os.walk(top, followlinks=True)   # Kaggle may mount inputs via symlinks

    def _version(code_root):
        import re as _re
        m = _re.search(r"__version__\s*=\s*(\d+)", (Path(code_root) / "src/t2s/__init__.py").read_text())
        return int(m.group(1)) if m else 0

    def _find_code():
        zips, found = [], []
        for root, dirs, files in _walk():
            if root.endswith("src/t2s") and "modeling.py" in files and not root.startswith(PROJECT_DIR):
                found.append(Path(root).parents[1])
        if found:   # several attached versions -> use the newest code
            found.sort(key=_version, reverse=True)
            if len(found) > 1:
                print("several code copies found:", [(str(f), _version(f)) for f in found])
            return found[0]
        for root, dirs, files in _walk():
            zips += [Path(root) / f for f in files if f.lower().endswith(".zip") and "text2sql" in f.lower()]
        for z in zips:   # dataset kept as an un-extracted zip -> extract it ourselves
            tmp = Path("/kaggle/tmp/code"); tmp.mkdir(parents=True, exist_ok=True)
            zipfile.ZipFile(z).extractall(tmp)
            for root, dirs, files in os.walk(tmp):
                if root.endswith("src/t2s") and "modeling.py" in files:
                    return Path(root).parents[1]
        return None

    code_root = _find_code()
    if code_root is None:
        print("Could not find the text2sql code. Contents of /kaggle/input:")
        for root, dirs, files in _walk():
            depth = root.count(os.sep) - 2
            if depth <= 4:
                print("  " * depth + os.path.basename(root) + "/", (files[:5] if files else ""))
        raise RuntimeError("Code dataset not attached. Right panel -> Input -> '+ Add Input' -> Your Work -> Datasets -> "
                           "text2sql-code -> Add. Then run again. (Send the listing above if it is attached.)")
    print("code found at", code_root, "| code version", _version(code_root))
    # Kaggle images ship torchao 0.10, which makes peft refuse to load LoRA adapters on fp16 models. We don't need it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True)
    shutil.copytree(code_root, PROJECT_DIR, dirs_exist_ok=True,   # always refresh the code
                    ignore=shutil.ignore_patterns("artifacts", "results", "__pycache__"))
    for m in [m for m in list(sys.modules) if m == "t2s" or m.startswith("t2s.")]:
        del sys.modules[m]   # pick up new code if the kernel was not restarted
    # Resume support: if a previous run's Output is attached as an input, reuse its results/artifacts
    for root, dirs, files in _walk():
        if os.path.basename(root) == "results" and os.path.basename(os.path.dirname(root)) == "text2sql":
            prev_root = Path(root).parent
            for sub_dir in ["results", "artifacts", "reports"]:
                if (prev_root / sub_dir).exists():
                    shutil.copytree(prev_root / sub_dir, f"{PROJECT_DIR}/{sub_dir}", dirs_exist_ok=True)
            print("restored previous results from", prev_root)
    if not SMOKE:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
elif SMOKE:
    PROJECT_DIR = os.environ["T2S_PROJECT_DIR"]
elif not ON_COLAB:
    # Local machine (VS Code / Jupyter): the notebook lives in text2sql/notebooks/
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/masters-projects/text2sql"   # <- change if needed
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART, FIG, RES = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures", f"{PROJECT_DIR}/results"
for d in (ART, FIG, RES): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from t2s.spider import download_spider, find_spider_root, load_split, db_path, serialize_schema, schema_stats
from t2s import modeling as MD, evaluate as EV
from t2s.analysis import hardness, HARDNESS_ORDER

# ---- data ----
if SMOKE:
    from t2s.smoke import make_fake_spider
    SPIDER = find_spider_root(make_fake_spider(f"{PROJECT_DIR}/data"))
else:
    if ON_KAGGLE:
        attached = [os.path.join(r, "dev.json") for r, d, f in os.walk("/kaggle/input", followlinks=True)
                    if "dev.json" in f and os.path.isdir(os.path.join(r, "database"))]   # Spider attached as input?
        SPIDER = find_spider_root(Path(attached[0]).parent) if attached else download_spider("/kaggle/working/spider")
    elif ON_COLAB:
        SPIDER = download_spider("/content/spider", drive_zip="/content/drive/MyDrive/masters-projects/spider_data.zip")
    else:
        SPIDER = download_spider(f"{PROJECT_DIR}/data/spider", drive_zip=f"{PROJECT_DIR}/data/spider_data.zip")
train_all, dev = load_split(SPIDER, "train"), load_split(SPIDER, "dev")

# ---- model ----
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
if SMOKE:
    from t2s.smoke import make_tiny_model
    tiny = f"{ART}/tiny-model"
    if not os.path.exists(f"{tiny}/config.json"):
        make_tiny_model(tiny, [e.question + " " + e.query for e in train_all] + [serialize_schema(db_path(SPIDER, e.db_id), "ddl_rows") for e in train_all[:20] + dev])
    BASE_MODEL = tiny
DEV_SUBSET = sorted(np.random.RandomState(0).choice(len(dev), min(300, len(dev)), replace=False))  # fixed subset for ablations
SEED = 42
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=float)
def load_json(name, default=None):
    p = f"{ART}/{name}.json"; return json.load(open(p)) if os.path.exists(p) else default
def cached(name, fn):
    """Run fn() once and store its DataFrame as results/<name>.parquet; later calls (after a crash) just load it."""
    p = f"{RES}/{name}.parquet"
    if os.path.exists(p):
        df = pd.read_parquet(p); print(f"[cached] {name}: EX={df.exec_match.mean():.4f}"); return df
    df = fn(); df.to_parquet(p, index=False); print(f"[done] {name}: EX={df.exec_match.mean():.4f}"); return df
def gen_cache(name):
    return f"{RES}/gen_{name}.jsonl"
print(f"train={len(train_all)}  dev={len(dev)}  model={BASE_MODEL}  device={MD.device_info()}")

In [ ]:
sys.path.insert(0, f"{PROJECT_DIR}/app")
from app import build_demo
run = load_json("main_run")
demo = build_demo(BASE_MODEL, run["adapter"], SPIDER, run["schema_format"], sorted({e.db_id for e in dev}))
demo.launch(share=not SMOKE, prevent_thread_lock=SMOKE)

## Publish the adapter (optional)
Add a Hugging Face *write* token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar), then set `HF_REPO`.

In [ ]:
PUBLISH = False
HF_REPO = "your-hf-username/qwen2.5-coder-1.5b-spider-qlora"
if PUBLISH and not SMOKE:
    from google.colab import userdata
    from huggingface_hub import login
    from peft import PeftModel
    login(userdata.get("HF_TOKEN"))
    m = MD.load_model(BASE_MODEL, load_4bit=False, adapter=run["adapter"])
    m.push_to_hub(HF_REPO); MD.load_tokenizer(BASE_MODEL).push_to_hub(HF_REPO)
    print("https://huggingface.co/" + HF_REPO)